# Playground Notebook for ISCO Data Modeling and Testing

**THIS IS AN EXPLORATORY RESOURCE, CODE HERE WON'T BE NECESSARY EFFICIENT AND THINGS COULD CHANGE WHILE THE ANALYSIS OR TEST IS BEING DEVELOPED**

## Environment

In [1]:
# Libraries 
import os
import pandas as pd
import io #To avoid writting in COSMOS, files will be loaded in Memory, i know, is not the best but won't be big data
import gzip
from dotenv import load_dotenv

In [2]:
# Variables
load_dotenv()

cosmos_path = os.getenv('COSMOS_PATH')
job_id = '48d95d3f-4ed7-4931-a113-aa4afb62e00f'

## Base Data

In [3]:
# Target file path definition
job_path = os.path.join(cosmos_path, 'raw', job_id)
file_path = os.path.join(job_path,os.listdir(job_path)[0]) # I know there is only one file there

# Decompression in Memory
with gzip.open(file_path, 'rb') as cf:
    decomp_data = cf.read()
    
# We're reading as Binary so, we need to wrap it to be read as excel
buffer = io.BytesIO(decomp_data)

# Now we're ready to read the file
data = pd.read_excel(buffer)
data.head()


,Level,ISCO 08 Code,Title EN,Definition,Tasks include,Included occupations,Excluded occupations,Notes
0,1,1,Managers,"Managers plan, direct, coordinate and evaluate...",Tasks performed by managers usually include: f...,Occupations in this major group are classified...,NaN,In distinguishing between managers classified ...
1,2,11,"Chief Executives, Senior Officials and Legisla...","Chief executives, senior officials and legisla...",Tasks performed by workers in this sub-major g...,Occupations in this sub-major group are classi...,NaN,NaN
2,3,111,Legislators and Senior Officials,"Legislators and senior officials determine, fo...",Tasks performed usually include: presiding ove...,Occupations in this minor group are classified...,NaN,NaN
3,4,1111,Legislators,"Legislators determine, formulate, and direct p...",Tasks include -\n(a) presiding over or partic...,Examples of the occupations classified here:\n...,NaN,NaN
4,4,1112,Senior Government Officials,Senior government officials advise governments...,"Tasks include -\n(a) advising national, state...",Examples of the occupations classified here:\n...,NaN,Chief executives of Government-owned enterpris...


## Transformations
In this case, we have a hierarchy in the data. This table could serve as the basis for a dictionary, but it might be worthwhile to have the code lineage in a single table, with codes listed side by side.

Another important consideration is that the occupations are stored in the variable Included Occupations. This should be extracted, labeled as its own sub-code, and related to its corresponding category. However, our actual roles ecosystem will include the following structure:

Occupation Group → Role → Title

Variables such as Definition and Task could be extremely valuable in the future, especially if we need to match titles or people's positions based on descriptions or tasks using LLM applications.
<br>
### First Table: roles_cluster

In this case we're renaming the Title EN or Occupation Group to roles_cluster, this mapping should live in our documentation and table metadata to be able to relate with the original concepts of the ILOSTAT. Using roles_cluster could drive a better understanding for non-users of ILOSTAT data.

Our first step will be separate the dict from the hierarchy and rename the variables to a simpler std way.

Before Starting a little parenthesis, Included and Excluded occupations, the first is coded and exclusions are related to those codes so Excluded occupations will be removed from the tables.

In [4]:
print('Included occupations:')
display(data[~data['Excluded occupations'].isna()][['Included occupations']].to_dict('records')[0])

print('\nExcluded occupations:')
display(data[~data['Excluded occupations'].isna()][['Excluded occupations']].to_dict('records')[0])

Included occupations:


{'Included occupations': 'Examples of the occupations classified here:\n- Company secretary\n- Finance manager'}


Excluded occupations:


{'Excluded occupations': 'Some related occupations classified elsewhere:\n- Financial institution branch manager - 1346\n- Financial controller - 2411\n- Management accountant - 2411'}

In [5]:
roles_dict = data.copy().drop(columns=['Included occupations','Excluded occupations'])
roles_cluster = data[['Level', 'ISCO 08 Code', 'Title EN']].rename(columns={'Level':'level',
                                                                            'ISCO 08 Code':'code',
                                                                            'Title EN':'name'})
roles_cluster['code'] = roles_cluster['code'].astype(str)
roles_cluster

,level,code,name
0,1,1,Managers
1,2,11,"Chief Executives, Senior Officials and Legisla..."
2,3,111,Legislators and Senior Officials
3,4,1111,Legislators
4,4,1112,Senior Government Officials
...,...,...,...
614,3,21,Non-commissioned Armed Forces Officers
615,4,210,Non-commissioned Armed Forces Officers
616,2,3,"Armed Forces Occupations, Other Ranks"
617,3,31,"Armed Forces Occupations, Other Ranks"


To create the code hierarchy (in columns) let's merge by levels bottom top

In [6]:
# Levels
roles_cluster.level.unique()

array([1, 2, 3, 4])

In [7]:
# levels datasets
l4_data = roles_cluster.loc[roles_cluster['level']==4].copy().drop(columns='level').rename(columns={'code':'code_l4',
                                                                                                    'name':'name_l4'})
l4_data['code_l3'] = l4_data['code_l4'].str[:-1]

l3_data = roles_cluster.loc[roles_cluster['level']==3].copy().drop(columns='level').rename(columns={'code':'code_l3',
                                                                                                    'name':'name_l3'})
l3_data['code_l2'] = l3_data['code_l3'].str[:-1]

l2_data = roles_cluster.loc[roles_cluster['level']==2].copy().drop(columns='level').rename(columns={'code':'code_l2',
                                                                                                    'name':'name_l2'})
l2_data['code_l1'] = l2_data['code_l2'].str[:-1]

l1_data = roles_cluster.loc[roles_cluster['level']==1].copy().drop(columns='level').rename(columns={'code':'code_l1',
                                                                                                    'name':'name_l1'})

In [8]:
roles_cluster = (
                 l4_data.
                 merge(l3_data, on='code_l3', how='left').
                 merge(l2_data, on='code_l2', how='left').
                 merge(l1_data, on='code_l1', how='left')
                 )

roles_cluster = roles_cluster[['code_l1','code_l2','code_l3','code_l4','name_l1','name_l2','name_l3','name_l4']]
roles_cluster

,code_l1,code_l2,code_l3,code_l4,name_l1,name_l2,name_l3,name_l4
0,1,11,111,1111,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Legislators
1,1,11,111,1112,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Senior Government Officials
2,1,11,111,1113,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Traditional Chiefs and Heads of Villages
3,1,11,111,1114,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Senior Officials of Special-interest Organizat...
4,1,11,112,1120,Managers,"Chief Executives, Senior Officials and Legisla...",Managing Directors and Chief Executives,Managing Directors and Chief Executives
...,...,...,...,...,...,...,...,...
431,9,96,962,9624,Elementary Occupations,Refuse Workers and Other Elementary Workers,Other Elementary Workers,Water and Firewood Collectors
432,9,96,962,9629,Elementary Occupations,Refuse Workers and Other Elementary Workers,Other Elementary Workers,Elementary Workers Not Elsewhere Classified
433,,1,11,110,NaN,Commissioned Armed forces Officers,Commissioned Armed Forces Officers,Commissioned Armed Forces Officers
434,,2,21,210,NaN,Non-commissioned Armed Forces Officers,Non-commissioned Armed Forces Officers,Non-commissioned Armed Forces Officers


### Second Table: roles

In this case we're renaming the Occupation to Role, this mapping should live in our documentation and table metadata to be able to relate with the original concepts of the ILOSTAT. In this case, we're renaming only because we want the Occupation would serve the same purpose as role in this case but we already declare Role in our tickets, so... let's imagine is a corporative data literacy concept.

The roles are held by the lowest level so we're going straightforward to extract that

In [9]:
# Base Data
roles = data.copy().drop(columns=['Definition','Title EN','Excluded occupations', 'Tasks include', 'Notes']).rename(columns={'Level':'level',
                                                                                                                             'ISCO 08 Code':'code',
                                                                                                                             'Included occupations':'role'})
roles = roles[roles['level']==4]
# Keeping only list of roles
roles['role'] = roles['role'].str.split('\n') # Splitting by \n 
roles['role'] = roles['role'].str[1:] # Removing introductory text
roles = roles.explode('role') # Explode <3
roles['role'] = roles['role'].str.strip().str.extract(r'([A-Za-z].*)')#.str.split('-').str[1:] # Cleaning names

# Cleaning the table
roles.drop(columns=['level'], inplace=True)
roles.rename(columns={'code':'code_l4',
                      'role':'role_name'}, inplace=True)
roles['code_l4'] = roles['code_l4'].astype(str) 
roles['role_code'] = None

roles.head()

,code_l4,role_name,role_code
3,1111,City councillor,None
3,1111,Government minister,None
3,1111,Mayor,None
3,1111,Member of parliament,None
3,1111,President (government),None


In [10]:
# Creating Codes
for cluster in roles['code_l4'].unique():
    filter_mask = roles['code_l4'] == cluster
    roles.loc[filter_mask, 'role_code'] = (
        roles.loc[filter_mask, 'code_l4'].astype(str) + 
        '-' + 
        (roles.loc[filter_mask].reset_index().index + 1).astype(str)
    )

roles

,code_l4,role_name,role_code
3,1111,City councillor,1111-1
3,1111,Government minister,1111-2
3,1111,Mayor,1111-3
3,1111,Member of parliament,1111-4
3,1111,President (government),1111-5
...,...,...,...
618,310,Gunner,310-6
618,310,Infantryman/woman,310-7
618,310,Paratrooper,310-8
618,310,Rifleman/woman,310-9


Now considering that all the roles will live in level 4, makes no sense to have a table for the clusters and another for this.

In [11]:
roles = roles_cluster.merge(roles, on='code_l4', how='left')
roles

,code_l1,code_l2,code_l3,code_l4,name_l1,name_l2,name_l3,name_l4,role_name,role_code
0,1,11,111,1111,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Legislators,City councillor,1111-1
1,1,11,111,1111,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Legislators,Government minister,1111-2
2,1,11,111,1111,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Legislators,Mayor,1111-3
3,1,11,111,1111,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Legislators,Member of parliament,1111-4
4,1,11,111,1111,Managers,"Chief Executives, Senior Officials and Legisla...",Legislators and Senior Officials,Legislators,President (government),1111-5
...,...,...,...,...,...,...,...,...,...,...
1958,,3,31,310,NaN,"Armed Forces Occupations, Other Ranks","Armed Forces Occupations, Other Ranks","Armed Forces Occupations, Other Ranks",Gunner,310-6
1959,,3,31,310,NaN,"Armed Forces Occupations, Other Ranks","Armed Forces Occupations, Other Ranks","Armed Forces Occupations, Other Ranks",Infantryman/woman,310-7
1960,,3,31,310,NaN,"Armed Forces Occupations, Other Ranks","Armed Forces Occupations, Other Ranks","Armed Forces Occupations, Other Ranks",Paratrooper,310-8
1961,,3,31,310,NaN,"Armed Forces Occupations, Other Ranks","Armed Forces Occupations, Other Ranks","Armed Forces Occupations, Other Ranks",Rifleman/woman,310-9


After seen this, then in the ETL all codes for "clusters" will be called occupation_code and names occupation_name, on the other hand role_name and role_code will prevail.

In [12]:
roles.columns

Index(['code_l1', 'code_l2', 'code_l3', 'code_l4', 'name_l1', 'name_l2',
       'name_l3', 'name_l4', 'role_name', 'role_code'],
      dtype='object')

## SQLAlchemy Model 

In [ ]:
from sqlalchemy import Column, Integer, Text, DateTime
from sqlalchemy.orm